# Network Flow V2 — Step 03: Three-dataset feature mapping (READ ONLY)

**Datasets:** Edge-IIoTset (`ML-EdgeIIoT-dataset.csv`), X-IIoTID (`X-IIoTID dataset.csv`), and ToN-IoT (`ton_iot_train_test_network.csv`).

**Goal:** Identify candidate equivalent columns, non-equivalent packet/flow measurements, source-specific telemetry, and label fields **before** splitting, cleaning or model training.

This notebook reads CSV headers and small samples only; it does **not** combine records, modify the original CSVs, split data, or train a model. Matching column names alone do **not** prove matching feature meanings, units or observation levels. These mappings are hypotheses requiring documentation and sample checks.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
FILES = {
    "edge_iiotset": ROOT / "data/raw/edge_iiotset/ML-EdgeIIoT-dataset.csv",
    "x_iiotid": ROOT / "data/raw/x_iiotid/X-IIoTID dataset.csv",
    "ton_iot": ROOT / "data/raw/ton_iot/ton_iot_train_test_network.csv",
}
LABEL_COLUMNS = {
    "edge_iiotset": ["Attack_label", "Attack_type"],
    "x_iiotid": ["class1", "class2", "class3"],
    "ton_iot": ["label", "type"],
}
REPORTS = ROOT / "reports" / "audits"
DOCS = ROOT / "docs"
for key, path in FILES.items():
    print(key, '|', 'FOUND' if path.is_file() else 'MISSING', '|', path)
assert all(path.is_file() for path in FILES.values()), "One or more files are missing; check ROOT or filenames."


## 1. Load actual headers (not full CSVs)

This gives an exact list of existing column names, which prevents relying on assumed dataset schemas.

In [ ]:
headers = {
    name: pd.read_csv(path, nrows=0).columns.tolist()
    for name, path in FILES.items()
}
for name, columns in headers.items():
    non_labels = [c for c in columns if c not in LABEL_COLUMNS[name]]
    print(f"{name}: {len(columns)} columns, {len(non_labels)} candidate non-label columns")
    missing_labels = set(LABEL_COLUMNS[name]) - set(columns)
    if missing_labels:
        raise ValueError(f"Missing expected label columns in {name}: {sorted(missing_labels)}")


## 2. Candidate feature mapping

**Interpretation of statuses:**
- `POTENTIAL_TON_X`: ToN-IoT and X-IIoTID have plausible corresponding features. **Not verified equivalent.**
- `PROTOCOL_SPECIFIC`: Edge-IIoTset has a packet/protocol field; it cannot be substituted directly for a connection-level field.
- `IDENTIFIER_ONLY`: Useful for grouping and investigations, but exclude raw IDs from first model inputs to reduce shortcut learning.
- `TIME_NOT_EQUIVALENT`: Different timestamp availability/semantics. Needs time-unit and ordering checks.
- `OTHER`: Conceptual/derived similarity only, not a direct matching column.

Mappings marked ToN/X candidates must be checked against release documentation, observation granularity, units, missing-value meaning and practical gateway extraction.

In [ ]:
# Format: concept, ToN-IoT, X-IIoTID, Edge-IIoTset, mapping status, explanation
candidate_rows = [
    ("source_ip", "src_ip", "Scr_IP", "ip.src_host", "IDENTIFIER_ONLY", "Identifier; possible capture/site shortcut; grouping metadata, not baseline X."),
    ("destination_ip", "dst_ip", "Des_IP", "ip.dst_host", "IDENTIFIER_ONLY", "Identifier; retain separately from learned behavioural features."),
    ("source_port", "src_port", "Scr_port", "tcp.srcport", "PROTOCOL_SPECIFIC", "ToN/X plausible; Edge field is TCP-packet scoped. Compare port ablation."),
    ("destination_port", "dst_port", "Des_port", "tcp.dstport", "PROTOCOL_SPECIFIC", "ToN/X plausible; Edge field is TCP-packet scoped."),
    ("network_protocol", "proto", "Protocol", None, "POTENTIAL_TON_X", "Edge has protocol-specific fields, but no verified generic protocol column."),
    ("service", "service", "Service", None, "POTENTIAL_TON_X", "Compare categorical definitions and placeholder conventions."),
    ("flow_duration", "duration", "Duration", None, "POTENTIAL_TON_X", "Verify units (e.g., seconds), handling of 0 and nonnumeric tokens."),
    ("source_payload_bytes", "src_bytes", "Scr_bytes", None, "POTENTIAL_TON_X", "Check directional meaning, payload vs total bytes, and unavailable tokens."),
    ("destination_payload_bytes", "dst_bytes", "Des_bytes", None, "POTENTIAL_TON_X", "Check directional meaning, payload vs total bytes, and unavailable tokens."),
    ("missed_bytes", "missed_bytes", "missed_bytes", None, "POTENTIAL_TON_X", "Check exporter meaning and when measurements are absent."),
    ("source_packets", "src_pkts", "Scr_pkts", None, "POTENTIAL_TON_X", "Packet count at connection level; no direct Edge packet-row equivalent."),
    ("destination_packets", "dst_pkts", "Des_pkts", None, "POTENTIAL_TON_X", "Packet count at connection level."),
    ("source_ip_bytes", "src_ip_bytes", "Scr_ip_bytes", None, "POTENTIAL_TON_X", "Despite name, candidate byte count; verify IP-header accounting."),
    ("destination_ip_bytes", "dst_ip_bytes", "Des_ip_bytes", None, "POTENTIAL_TON_X", "Despite name, candidate byte count; verify IP-header accounting."),
    ("connection_state", "conn_state", "Conn_state", None, "POTENTIAL_TON_X", "Check state vocabulary and producer-specific meanings."),
    ("tcp_syn_signal", None, "is_syn_only", "tcp.connection.syn", "PROTOCOL_SPECIFIC", "One field may be a flow flag, the other packet/protocol indicator; not interchangeable."),
    ("packet_or_capture_time", None, "Timestamp", "frame.time", "TIME_NOT_EQUIVALENT", "ToN selected CSV has no time field; verify timestamp origin and units."),
    ("dns_query", "dns_query", None, "dns.qry.name", "PROTOCOL_SPECIFIC", "Different recording granularity; do not equate raw DNS values."),
    ("http_method", "http_method", None, "http.request.method", "PROTOCOL_SPECIFIC", "May differ by protocol parsing and aggregation."),
    ("tcp_payload_length", None, None, "tcp.len", "OTHER", "This is NOT the same measurement as directional flow byte counts."),
]
map_df = pd.DataFrame(candidate_rows, columns=[
    "concept", "ton_iot", "x_iiotid", "edge_iiotset", "review_status", "why_review"
])
for dataset in FILES:
    map_df[f"{dataset}_column_exists"] = map_df[dataset].apply(
        lambda v: isinstance(v, str) and v in headers[dataset]
    )
display(map_df[[
    "concept", "ton_iot", "ton_iot_column_exists", "x_iiotid",
    "x_iiotid_column_exists", "edge_iiotset", "edge_iiotset_column_exists", "review_status"
]])
print("These are mapping candidates, NOT verified common features.")


## 3. Review potential problems

1. A feature in one dataset might be **one packet** and in another an **entire connection**. Those cannot be directly combined.
2. A string value `-` or `?` is potentially unavailable, **not** necessarily numeric 0.
3. Never use label fields as model inputs (`Attack_label`, `Attack_type`, `class1`, `class2`, `class3`, `label`, `type`).
4. Source/destination IPs, raw payloads, exact timestamps and port combinations can create shortcut or leakage risks.
5. For a Raspberry Pi demonstration, prioritize features that your live extractor could compute, not merely those reported in a published CSV.
6. Recheck duplicate fingerprints **after** final feature selection and keep related sessions/campaigns together when splitting.

In [ ]:
# Brief, read-only token profiling from the first 5,000 rows per dataset.
# This sample is for inspecting representation only: DO NOT report these as full-data frequencies.
interesting = {
    "edge_iiotset": ["frame.time", "tcp.srcport", "tcp.dstport", "tcp.len"],
    "x_iiotid": ["Timestamp", "Duration", "Scr_bytes", "Des_bytes", "Scr_pkts", "is_with_payload"],
    "ton_iot": ["duration", "src_bytes", "dst_bytes", "src_pkts", "service", "conn_state"],
}
unknown_markers = {"", "-", "?", "na", "n/a", "nan", "null", "none", "unknown", "inf", "-inf"}
inspection = []
for dataset, path in FILES.items():
    cols = [c for c in interesting[dataset] if c in headers[dataset]]
    sample = pd.read_csv(path, nrows=5000, usecols=cols, dtype="string", keep_default_na=False)
    for col in cols:
        stripped = sample[col].fillna("").str.strip()
        lowered = stripped.str.lower()
        counts = lowered.value_counts(dropna=False)
        inspection.append({
            "dataset": dataset,
            "column": col,
            "sample_rows": len(sample),
            "placeholder_rows_in_sample": int(lowered.isin(unknown_markers).sum()),
            "distinct_values_in_sample": int(stripped.nunique()),
            "common_token_counts_in_sample": "; ".join(
                f"{repr(str(k))}:{int(v)}" for k, v in counts.head(3).items()
            ),
        })
print("ONLY a first-5,000-row representation check — may be class-order biased.")
display(pd.DataFrame(inspection))


## 4. Save mapping reports

Saves only a small Markdown and JSON **draft**, not raw data or train/test partitions. Fields remain unverified until the source documentation and actual semantics have been checked.

In [ ]:
REPORTS.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "status": "DRAFT — column presence confirmed; semantic equivalence NOT established",
    "files": {key: {"name": path.name, "size_bytes": path.stat().st_size} for key, path in FILES.items()},
    "columns": {key: len(val) for key, val in headers.items()},
    "target_labels": LABEL_COLUMNS,
    "candidate_mappings": map_df.where(pd.notna(map_df), None).to_dict(orient="records"),
    "notes": [
        "Do not append/merge the datasets based on this mapping.",
        "Prove feature definitions and units before claiming equivalent measurements.",
        "Edge-IIoTset CSV is packet/protocol oriented; ToN and X-IIoTID are more connection-oriented.",
        "Feature availability during live Raspberry Pi capture remains unverified.",
        "No rows were cleaned or split by this notebook.",
    ],
}
json_file = REPORTS / "three_dataset_feature_mapping_draft_v1.json"
json_file.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

def markdown_table(df):
    names = ["Concept", "ToN-IoT", "X-IIoTID", "Edge-IIoTset", "Status"]
    cols = ["concept", "ton_iot", "x_iiotid", "edge_iiotset", "review_status"]
    header = "| " + " | ".join(names) + " |"
    divider = "| " + " | ".join(["---"] * len(names)) + " |"
    entries = [header, divider]
    for _, row in df[cols].iterrows():
        entries.append("| " + " | ".join(str(row[c]) if pd.notna(row[c]) else "—" for c in cols) + " |")
    return "\n".join(entries)

md_file = DOCS / "THREE_DATASET_FEATURE_MAPPING_DRAFT.md"
text = "\n".join([
    "# Three-dataset feature mapping — DRAFT",
    "",
    "**Warning:** Similar names do not prove equivalent measurement units, granularity or meaning.",
    "**No three-way unified feature set has been verified.**",
    "",
    markdown_table(map_df),
    "",
    "## Requirements before training",
    "- Verify published feature meanings and units.",
    "- Verify X-IIoTID missing-token representations in numeric traffic/host data.",
    "- Separate network from host/process/sensor features.",
    "- Select deployment-observable features and recompute feature fingerprints.",
    "- Reserve scenario/time/group-separated validation/test partitions where supported.",
    "- Compare per-dataset baselines before compatible cross-dataset testing.",
    "",
])
md_file.write_text(text, encoding="utf-8")
print('Saved:', json_file)
print('Saved:', md_file)
print('Raw data unchanged; no split or training performed.')
